# Dryden Turbulence Model - Metric

In [ ]:
# Imports
import math
import numpy as np
import scipy as sp
from scipy.signal import lti
from scipy.interpolate import interp2d
import matplotlib.pyplot as plt

In [ ]:
# Constants
M_TO_FT = 3.28084
FT_TO_M = 1 / M_TO_FT

In [ ]:
# given
wingspan_m = 3.96

pose = {'position_m':[150, 22, 457.2]}
velocity_m_per_s = np.array([9.85, -0.72, 0.12])

In [ ]:
LAUNCH_TURBULENCE_INTENSITY_LIGHT_mpers       = 7.71667 # 15[kts]
LAUNCH_TURBULENCE_INTENSITY_MODERATE_mpers    = 15.4333 # 30[kts] 
LAUNCH_TURBULENCE_INTENSITY_SEVER_mpers       = 23.15 # 45[kts] 

launch_turbulence_intensity = LAUNCH_TURBULENCE_INTENSITY_MODERATE_mpers

In [ ]:
# noise_file = 'data/noise_with_scale_uvw.csv'
noise_file = 'data/noise_without_scale_uvw.csv'

my_data = np.genfromtxt(noise_file, delimiter=',')

t_s = my_data[:,0]
noise_u = my_data[:,1]
noise_v = my_data[:,2]
noise_w = my_data[:,3]

sim_start_s = t_s[0]
sim_end_s = t_s[-1]
dt_s = t_s[1] - t_s[0]

noise_scale = np.sqrt(math.pi / dt_s)    

In [ ]:
# Probability of Exceedance Table
# Figure 7 from p. 49 of MIL-F-8785C


poe_altitude_ft = np.array([500.0, 1750.0, 3750.0, 7500.0, 15000.0, 25000.0, 35000.0, 45000.0, 55000.0, 65000.0, 75000.0, 80000.0])

poe_index = np.array([1,2,3,4,5,6,7]) # 3: Light, 4: Moderate, 6: Severe
poe_table = np.array([
    [ 3.2,   2.2,   1.5,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 4.2,   3.6,   3.3,   1.6,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 6.6,   6.9,   7.4,   6.7,   4.6,   2.7,   0.4,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 8.6,   9.6,  10.6,  10.1,   8.0,   6.6,   5.0,   4.2,   2.7,   0.0,  0.0,  0.0],
    [11.8,  13.0,  16.0,  15.1,  11.6,   9.7,   8.1,   8.2,   7.9,   4.9,  3.2,  2.1],
    [15.6,  17.6,  23.0,  23.6,  22.1,  20.0,  16.0,  15.1,  12.1,   7.9,  6.2,  5.1],
    [18.7,  21.5,  28.4,  30.2,  30.7,  31.0,  25.2,  23.1,  17.5,  10.7,  8.4,  7.2]
])

poe_lookup = interp2d(poe_altitude_ft, poe_index, poe_table)

In [ ]:
poe_altitude_m = FT_TO_M * poe_altitude_ft
poe_m_table = FT_TO_M * poe_table

poe_m_lookup = interp2d(poe_altitude_m, poe_index, poe_m_table)

print(poe_m_table)

In [ ]:
altitude_m = pose['position_m'][2]

# def get_scale_length_and_intensities(altitude_ft, wingspan_ft):

#     scale_length = dict('')
#     intensity = dict()

if altitude_m < 304.8:
    scale_length_longitude  = altitude_m / (0.177 + 0.0027*altitude_m)**1.2
    scale_length_lateral    = scale_length_longitude
    scale_length_vertical   = altitude_m

    intensity_longitude = 0.06 * launch_turbulence_intensity / (0.177 + 0.0027*altitude_m)**0.4
    intensity_lateral   = intensity_longitude
    intensity_vertical  = 0.06 * launch_turbulence_intensity

elif altitude_m > 609.6:
    scale_length_longitude  = 762 # 2500[FT] * 0.30478 [M/FT]
    scale_length_lateral    = 762 # 2500[FT] * 0.30478 [M/FT]
    scale_length_vertical   = 762 # 2500[FT] * 0.30478 [M/FT]

    intensity_longitude     = poe_m_lookup(altitude_m, 4.0)
    intensity_lateral       = intensity_longitude
    intensity_vertical      = intensity_longitude

else:
    low = 304.8 / (0.177 + 0.0027*304.8)**1.2
    high = 762

    print('low', low)
    print('high', high)

    intensity_vert_low = 0.06 * launch_turbulence_intensity
    intensity_long_low = 0.06 * launch_turbulence_intensity / (0.177 + 0.0027*304.8)**0.4

    intensity_high     = poe_m_lookup(609.6, 4.0)[0]

    scale_length_longitude  = np.interp(altitude_m, [304.8, 609.6], [low, high])
    scale_length_lateral    = np.interp(altitude_m, [304.8, 609.6], [low, high])
    scale_length_vertical   = np.interp(altitude_m, [304.8, 609.6], [304.8, high])

    intensity_longitude     = np.interp(altitude_m, [304.8, 609.6], [intensity_long_low, intensity_high])
    intensity_lateral       = np.interp(altitude_m, [304.8, 609.6], [intensity_long_low, intensity_high])
    intensity_vertical      = np.interp(altitude_m, [304.8, 609.6], [intensity_vert_low, intensity_high])



print('Scale Length, Longitude: %f' % scale_length_longitude)
print('Scale Length, Lateral: %f'   % scale_length_lateral)
print('Scale Length, Vertical: %f'  % scale_length_vertical)

print('Intensity, Longitude: %f' % intensity_longitude)
print('Intensity, Lateral: %f'   % intensity_lateral)
print('Intensity, Vertical: %f'  % intensity_vertical)

In [ ]:
# linear rate filter
# given
# - sccale lengths
# - intensities
# - Velocity

# velocity_ft_per_s = velocity_m_per_s * M_TO_FT
velocity = np.linalg.norm(velocity_m_per_s)

velocity_scale_longitude    = scale_length_longitude / velocity
velocity_scale_lateral      = scale_length_lateral / velocity
velocity_scale_vertical     = scale_length_vertical / velocity
print(velocity_scale_longitude)
print(velocity_scale_lateral  )
print(velocity_scale_vertical )
# numerators denoted with 'b'
# denominators denoted with 'a'

b_u     = intensity_longitude * math.sqrt(2 * velocity_scale_longitude / math.pi)
b0_u    = b_u * 1
as_u    = velocity_scale_longitude
a0_u    = 1

b_v     = intensity_lateral * math.sqrt(velocity_scale_lateral / math.pi)
bs_v    = b_v * math.sqrt(3.0) * velocity_scale_lateral
b0_v    = b_v
as2_v   = velocity_scale_lateral ** 2
as_v    = 2 * velocity_scale_lateral
a0_v    = 1

b_w     = intensity_vertical * math.sqrt(velocity_scale_vertical / math.pi)
bs_w    = b_w * math.sqrt(3.0) * velocity_scale_vertical
b0_w    = b_w
as2_w   = velocity_scale_vertical ** 2
as_w    = 2 * velocity_scale_vertical
a0_w    = 1

H_u = lti(  [b0_u],
            [as_u, a0_u])

H_v = lti( [bs_v, b0_v], 
           [as2_v, as_v, a0_v])

H_w = lti( [bs_w, b0_w], 
           [as2_w, as_w, a0_w])

print(H_u)
print(H_v)
print(H_w)

In [ ]:
turbulence_linear_rate = np.array([
                                    H_u.output(noise_scale * noise_u, t_s)[1],
                                    H_v.output(noise_scale * noise_v, t_s)[1],
                                    H_w.output(noise_scale * noise_w, t_s)[1]])



plt.plot(t_s, turbulence_linear_rate[0])
plt.ylabel('linear rate - longitude (m/s)')
plt.xlabel('time(s)')

In [ ]:
linear_rate_u = [0]
linear_rate_v = [0] 
linear_rate_w = [0]

a_u = (1 - velocity/scale_length_longitude * dt_s)
b_u = intensity_longitude * math.sqrt(2 * velocity / scale_length_longitude * dt_s)

a_v = (1  - 2 * velocity/scale_length_lateral * dt_s)
b_v = intensity_lateral * math.sqrt(4 * velocity / scale_length_lateral * dt_s)

a_w = (1  - 2 * velocity/scale_length_vertical * dt_s)
b_w = intensity_vertical * math.sqrt(4 * velocity / scale_length_vertical * dt_s)


for step in range(1,len(t_s)):

    rate_u =  linear_rate_u[step-1] *  a_u \
                + noise_u[step-1] * b_u

    rate_v =  linear_rate_v[step - 1] *  a_v \
                + noise_v[step-1] * b_v

    rate_w =  linear_rate_w[step - 1] *  a_w \
                + noise_w[step-1] * b_w


    linear_rate_u.append(rate_u)
    linear_rate_v.append(rate_v)
    linear_rate_w.append(rate_w)

plt.plot(t_s, linear_rate_u)
plt.ylabel('linear rate - longitude (m/s)')
plt.xlabel('time(s)')

In [ ]:
plt.plot(t_s, linear_rate_w)
plt.ylabel('linear rate - longitude (m/s)')
plt.xlabel('time(s)')

In [ ]:
# Comparing in ft/s
plt.plot(t_s, M_TO_FT * turbulence_linear_rate[0])
plt.ylabel('linear rate - longitude (ft/s)')
plt.xlabel('time(s)')

In [ ]:
# PSD Analysis from discrete solution:
freq_sample = 1 / dt_s # sample rate, number of samples per unit of time

nblock = int(len(t_s) / 50)
overlap = int(nblock / 2)   
win = sp.signal.windows.hann(nblock, False)

SF = np.linalg.norm(win)**2 / np.sum(win)**2

In [ ]:
Npts = 200
freq_theo = np.logspace(-2, 2, Npts)
w = 2.0 * math.pi * freq_theo

In [ ]:
Ku = intensity_longitude**2 * velocity_scale_longitude / math.pi
psd_theo_u = Ku / (1 + (velocity_scale_longitude * w)**2)
psd_theo_u = 10 * np.log10(psd_theo_u)


In [ ]:
freq_sim_u, psd_sim_u = sp.signal.welch(linear_rate_u, freq_sample, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)
psd_sim_u = 10*np.log10(psd_sim_u)

plt.semilogx(freq_sim_u, psd_sim_u, '.', label='Simulated')
plt.semilogx(freq_theo, psd_theo_u, label='Theoretical Dryden')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])


# plt.plot(time_s, rate_u)
plt.show()

# Observing combined behavior

In [ ]:
plt.plot(t_s, np.sqrt(np.square(linear_rate_u) + np.square(linear_rate_v) + np.square(linear_rate_w)))
plt.ylabel('linear rate - magnitude (m/s)')
plt.xlabel('time(s)')